# Telco Customer Churn Prediction with ML Lifecycle

## 1) Business Problem Definition

This is a binary classification problem: predict whether a customer will churn. The business goal is to identify at-risk customers early, launch retention strategies, and reduce churn while protecting revenue.

## 2) Data Collection

The dataset is the Telco Customer Churn dataset, stored as a CSV file. We will read it into a pandas DataFrame and inspect the schema before modeling.

## 3) Data Understanding & EDA

We inspect the shape, target distribution, column types, and basic statistics to understand data quality and identify anomalies before training.

## 4) Data Cleaning & Preprocessing

We standardize data types, handle missing values, and transform the target variable to a binary format suitable for machine learning.

## 5) Feature Engineering

We create useful derived features such as tenure groups and ratio-based features to improve model learning and business interpretability.

## 6) Feature Selection

We reduce the feature set to the most useful variables using a statistical selector to improve performance and reduce noise.

## 7) Train / Validation / Test Split

We split the dataset into train, validation, and test sets while preserving the class distribution for reliable evaluation.

## 8) Model Selection

We compare three models: XGBoost, CatBoost, and Random Forest. These are strong classifiers for tabular data and suitable for churn prediction.

## 9) Model Training

We train each model using the train set and assess validation performance to choose the right approach.

## 10) Hyperparameter Tuning

We search for stronger parameters using cross-validation and ROC-AUC as the optimization metric.

## 11) Model Evaluation & Validation

After tuning, we check precision, recall, F1, accuracy, ROC-AUC, and log loss on the test set to validate the final choices.

## 12) Model Interpretation

We inspect which features matter most to understand model behavior and explain churn drivers to stakeholders.

## 13) Model Packaging

We save the trained model and metadata so it can be reused in production and versioned safely.

## 14) Model Deployment

A deployment-ready artifact is loaded and used to score new data in a simple production-like workflow.

## 15) Production Monitoring

We track churn probability, model output distribution, and label drift to detect model degradation after launch.

## 16) Model Retraining

When performance drops or data changes significantly, the model can be retrained with the latest data.

## 17) Model Versioning & Governance

We document the model version, training snapshot, and governance checks so the system stays auditable and explainable.

## 18) Continuous Improvement / Retirement

Models need to be monitored, retrained, or retired when performance no longer meets the required business thresholds.

## Final Summary

This notebook shows the end-to-end ML lifecycle for the telco churn use case using XGBoost, CatBoost, and Random Forest. The final production model can then be packaged, deployed, monitored, and retrained systematically.

In [2]:
# Step 1: Business Problem Definition
from pathlib import Path

target_col = "Churn"
problem_type = "binary_classification"
business_goal = "Reduce customer churn by identifying customers likely to leave"
print(f"Target variable: {target_col}")
print(f"Problem type: {problem_type}")
print(f"Business goal: {business_goal}")


Target variable: Churn
Problem type: binary_classification
Business goal: Reduce customer churn by identifying customers likely to leave


In [ ]:
# Step 2: Data Collection
import pandas as pd

file_path = Path(r"d:\Project\customer-churn\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(file_path)
print(f"Dataset shape: {df.shape}")
print(df.head(3).to_string(index=False))


In [8]:
# Step 3: Data Understanding & EDA
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [13]:
print("\nDescriptive statistics:\n")
print(df.describe(include="all").transpose())


Descriptive statistics:

                   count unique               top  freq       mean        std  \
customerID          7043   7043        7590-VHVEG     1        NaN        NaN   
gender              7043      2              Male  3555        NaN        NaN   
SeniorCitizen     7043.0    NaN               NaN   NaN   0.162147   0.368612   
Partner             7043      2                No  3641        NaN        NaN   
Dependents          7043      2                No  4933        NaN        NaN   
tenure            7043.0    NaN               NaN   NaN  32.371149  24.559481   
PhoneService        7043      2               Yes  6361        NaN        NaN   
MultipleLines       7043      3                No  3390        NaN        NaN   
InternetService     7043      3       Fiber optic  3096        NaN        NaN   
OnlineSecurity      7043      3                No  3498        NaN        NaN   
OnlineBackup        7043      3                No  3088        NaN        NaN   
De

In [ ]:
print("\nTarget distribution:\n")
print(df["Churn"].value_counts(normalize=True))

In [ ]:
print("\nMissing values by column:\n")
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Step 4: Data Cleaning & Preprocessing
import numpy as np
import pandas as pd

clean_df = df.copy()
clean_df["TotalCharges"] = pd.to_numeric(clean_df["TotalCharges"], errors="coerce")
clean_df["Churn"] = clean_df["Churn"].map({"Yes": 1, "No": 0})
clean_df = clean_df.drop(columns=["customerID"], errors="ignore")

numeric_cols = clean_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col == "Churn":
        continue
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

cat_cols = clean_df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].mode().iloc[0])

print("Missing values after cleaning:", clean_df.isna().sum().sum())
print(clean_df.head(3).to_string(index=False))


In [ ]:
# Step 5: Feature Engineering
import numpy as np
import pandas as pd

feat_df = clean_df.copy()
feat_df["tenure_group"] = pd.cut(
    feat_df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"],
    right=False,
)
feat_df["MonthlyToTotalRatio"] = np.where(
    feat_df["TotalCharges"] != 0,
    feat_df["MonthlyCharges"] / feat_df["TotalCharges"],
    0,
)
feat_df["is_long_term"] = (feat_df["tenure"] >= 24).astype(int)
feat_df = pd.get_dummies(feat_df, drop_first=False)
print(f"Feature-engineered data shape: {feat_df.shape}")
print(feat_df.head(3).to_string(index=False))


In [ ]:
# Step 6: Feature Selection
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif

X = feat_df.drop(columns=["Churn"])
y = feat_df["Churn"]
selector = SelectKBest(score_func=mutual_info_classif, k=min(25, X.shape[1]))
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]
X = pd.DataFrame(X_selected, columns=selected_features, index=X.index)
print(f"Selected feature count: {len(selected_features)}")
print(selected_features[:15].tolist())


In [ ]:
# Step 7: Train / Validation / Test Split
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)
print(f"Train shape: {X_train.shape}, y_train: {y_train.shape}")
print(f"Validation shape: {X_val.shape}, y_val: {y_val.shape}")
print(f"Test shape: {X_test.shape}, y_test: {y_test.shape}")


In [ ]:
# Step 8: Model Selection
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
    ),
    "CatBoost": CatBoostClassifier(
        loss_function="Logloss",
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
    ),
}
print("Selected models:")
for name in models:
    print(f"- {name}")


In [ ]:
# Step 9: Model Training
from sklearn.metrics import accuracy_score, roc_auc_score

training_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    val_pred = (val_prob >= 0.5).astype(int)
    training_results[name] = {
        "model": model,
        "val_prob": val_prob,
        "val_pred": val_pred,
    }
    print(f"{name} - validation ROC AUC: {roc_auc_score(y_val, val_prob):.4f}")
    print(f"{name} - validation accuracy: {accuracy_score(y_val, val_pred):.4f}")


In [ ]:
# Step 10: Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

xgb_search = RandomizedSearchCV(
    XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42),
    param_distributions={
        "n_estimators": [200, 400],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    },
    scoring="roc_auc",
    n_iter=8,
    cv=3,
    random_state=42,
    n_jobs=1,
)
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions={
        "n_estimators": [200, 400],
        "max_depth": [None, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    },
    scoring="roc_auc",
    n_iter=8,
    cv=3,
    random_state=42,
    n_jobs=1,
)
cat_search = RandomizedSearchCV(
    CatBoostClassifier(loss_function="Logloss", random_seed=42, verbose=False),
    param_distributions={
        "iterations": [200, 400],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5],
    },
    scoring="roc_auc",
    n_iter=5,
    cv=3,
    random_state=42,
    n_jobs=1,
)

xgb_search.fit(X_train, y_train)
rf_search.fit(X_train, y_train)
cat_search.fit(X_train, y_train)

print("XGB best params:", xgb_search.best_params_)
print("Random Forest best params:", rf_search.best_params_)
print("CatBoost best params:", cat_search.best_params_)


In [ ]:
# Step 11: Model Evaluation & Validation
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
)

best_models = {
    "XGBoost": xgb_search.best_estimator_,
    "RandomForest": rf_search.best_estimator_,
    "CatBoost": cat_search.best_estimator_,
}

results = []
for name, model in best_models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "LogLoss": log_loss(y_test, prob),
    }
    results.append(metrics)
    print(f"\n{name}\n")
    print(pd.DataFrame([metrics]).to_string(index=False))
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

model_summary = pd.DataFrame(results)
print("\nModel summary:\n")
print(model_summary.sort_values("ROC_AUC", ascending=False).to_string(index=False))


In [ ]:
# Step 12: Model Interpretation
import pandas as pd

for name, model in best_models.items():
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif hasattr(model, "get_feature_importance"):
        importances = model.get_feature_importance()
    else:
        continue

    feature_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    print(f"\nTop 10 important features for {name}:\n")
    print(feature_importance.head(10).to_string(index=False))


In [ ]:
# Step 13: Model Packaging
import json
import joblib
from datetime import datetime
from pathlib import Path

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
selected_model = best_models["XGBoost"]
model_path = artifacts_dir / f"telco_churn_xgboost_{timestamp}.pkl"
joblib.dump(selected_model, model_path)

metadata = {
    "model_name": "telco_churn_xgboost",
    "created_at": timestamp,
    "target_column": "Churn",
    "train_rows": X_train.shape[0],
    "validation_rows": X_val.shape[0],
    "test_rows": X_test.shape[0],
    "selected_features": X.columns.tolist(),
}
with open(artifacts_dir / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to: {model_path}")
print(f"Saved metadata to: {artifacts_dir / 'model_metadata.json'}")


In [ ]:
# Step 14: Model Deployment
import joblib

production_model_path = Path("artifacts") / "production_model.pkl"
joblib.dump(best_models["XGBoost"], production_model_path)
loaded_model = joblib.load(production_model_path)

sample_rows = X_test.iloc[:5].copy()
probabilities = loaded_model.predict_proba(sample_rows)[:, 1]
print("Deployment prediction example:\n")
print(pd.DataFrame({
    "customer_index": sample_rows.index,
    "churn_probability": probabilities,
}).round(4).to_string(index=False))


In [ ]:
# Step 15: Production Monitoring
import pandas as pd

monitor_df = pd.DataFrame({
    "actual": y_test.reset_index(drop=True).iloc[:200].values,
    "predicted_probability": best_models["XGBoost"].predict_proba(X_test.iloc[:200])[:, 1],
})
monitor_df["predicted_label"] = (monitor_df["predicted_probability"] >= 0.5).astype(int)
monitor_df["error"] = monitor_df["actual"] - monitor_df["predicted_label"]

print(monitor_df.head().to_string(index=False))
print(f"\nAverage churn probability: {monitor_df['predicted_probability'].mean():.4f}")
print(f"Average predicted churn rate: {monitor_df['predicted_label'].mean():.4f}")


In [ ]:
# Step 16: Model Retraining
from xgboost import XGBClassifier

retrained_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)

X_retrain = pd.concat([X_train, X_val], axis=0)
y_retrain = pd.concat([y_train, y_val], axis=0)
retrained_model.fit(X_retrain, y_retrain)
retrained_prob = retrained_model.predict_proba(X_test)[:, 1]
retrained_auc = roc_auc_score(y_test, retrained_prob)
print(f"Updated model ROC AUC on test set: {retrained_auc:.4f}")


In [ ]:
# Step 17: Model Versioning & Governance
import json
from pathlib import Path

version_info = {
    "model_version": "v1.0.0",
    "model_family": "xgboost",
    "training_date": "2026-09-21",
    "owner": "data_science_team",
    "target": "Churn",
    "validation_metric": {"roc_auc": round(retrained_auc, 4)},
    "governance_checks": [
        "Train/validation/test split preserved",
        "Feature selection logged",
        "Artifacts saved with metadata",
        "Model retraining tracked",
    ],
}

with open(Path("artifacts") / "model_version.json", "w", encoding="utf-8") as file:
    json.dump(version_info, file, indent=2)

print(version_info)


In [ ]:
# Step 18: Continuous Improvement / Retirement
threshold = 0.80
if retrained_auc >= threshold:
    print(f"Model remains active. Test ROC AUC = {retrained_auc:.4f} exceeds retirement threshold {threshold:.4f}.")
else:
    print(f"Model should be retrained or retired. Test ROC AUC = {retrained_auc:.4f} is below threshold {threshold:.4f}.")
print("Next improvement ideas: add more customer-level behavioral features, test other thresholds, and monitor drift.")


In [ ]:
# Final summary
print("\nCompleted telco churn ML lifecycle notebook with XGBoost, CatBoost, and Random Forest.")
print("This workflow covers data ingestion, preprocessing, feature engineering, model selection, tuning, validation, packaging, deployment, monitoring, retraining, and governance.")
